# NB01 — Do prompt ao agente

**Pergunta:** o que separa um prompt de um agente?  
**Caminho:** `v0` (agente sem ferramentas, com a política colada) → `v1` (ferramentas, decisão estruturada, estado, contexto enxuto, memória).

Cada seção mostra um problema real do RangoJá e a correção. As execuções vêm de **gravações do modelo real**; quando uma gravação foi escolhida entre várias, a célula diz a taxa.

> ▶ **Comece por aqui:** execute a célula de código abaixo **uma vez**, antes de qualquer seção. Ela importa tudo que os exemplos usam (`atender`, `verificar`, …). Depois disso, cada seção roda sozinha, na ordem que você quiser.

In [1]:
# Execute esta célula primeiro: ela importa o que todas as seções usam.
from rangoja import config, dados, falhas, ferramentas, controle, observabilidade, relogio
from rangoja.armazem import Armazem
from rangoja.casos import caso
from rangoja.execucao import atender, continuar, aprovar, recusar, expirar_pendencias, reiniciar_runner, estado_persistido
from rangoja.notebook import (preparar_ambiente, preparar_secao, mostrar, comparar, verificar, verificar_falha,
                              resumo, amostra, matriz)
from rangoja.trajetoria import chamou, ferramenta_antes, contar_chamadas
from rangoja.versoes import v0, v1, v2, v3, montar
preparar_ambiente()

modo,replay
modelo,gpt-4o-mini
chave de API,não (use replay)
observabilidade,indisponível (ModuleNotFoundError); usando anatomia em texto


## N1.1 — O prompt que funciona na demo

Antes de qualquer atalho, o agente inteiro, em ADK puro. Primeiro, o que chegou: a reclamação do cliente e o pedido como ele está no sistema.

In [2]:
mensagem = "Meu pedido 4821 chegou frio e faltou o refri."

pedido = dados.pedido("4821")
for item in pedido.itens:
    print(f"{item.nome:22s} R$ {float(item.preco):6.2f}   {'entregue' if item.entregue else 'NÃO entregue'}")
print(f"{'total':22s} R$ {float(pedido.total):6.2f}")

Hambúrguer artesanal   R$  38.00   entregue
Batata média           R$  18.00   entregue
Refrigerante 2L        R$   8.00   NÃO entregue
total                  R$  70.00


O agente é um `LlmAgent` do ADK: um nome, um modelo e uma instrução. Aqui a instrução é o prompt com a política inteira colada. O modelo vem do `.env`: em `live`, o `LiteLlm` chama o modelo de `RANGOJA_MODELO` com a sua chave; em `replay`, uma gravação do modelo real responde no lugar dele.

As linhas comentadas são os **pontos de intervenção**: os lugares onde o código pode agir sobre o agente. Cada seção do material preenche um deles.

In [3]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from rangoja.notebook import gravacao

politica = "\n\n".join(f"## {tema}\n{texto}" for tema, texto in dados.politica_secoes().items())
PROMPT = ("Você atende clientes do RangoJá, um app de delivery. Responda em português, com cordialidade e objetividade."
          "\n\nPolítica de atendimento:\n" + politica)

if config.modo() == "replay":
    llm = gravacao("N1.1/falha")  # sem chave: o que o modelo real respondeu quando gravamos
else:
    llm = LiteLlm(model=f"openai/{config.modelo_primario()}")  # o modelo de RANGOJA_MODELO (.env), com a sua chave

agente = LlmAgent(
    name="unico",
    model=llm,
    instruction=PROMPT,
    # tools=[buscar_pedido, registrar_reembolso, ...]   ← N1.2  ferramentas trazem fatos e agem
    # output_schema=Decisao                             ← N1.4  decisão que o código consegue ler
    # before_model_callback=guardrail_entrada           ← N2.2  filtra o que chega ao modelo
    # after_model_callback=guardrail_saida              ← N2.3  filtra o que sai para o cliente
    # before_tool_callback=limite_ferramenta            ← N2.4, N3.4  autoriza cada ação
    # on_tool_error_callback=erro_estruturado           ← N3.2  erro de ferramenta vira resultado
)
print(f"modelo: {'gravação N1.1/falha' if config.modo() == 'replay' else config.modelo_primario() + ' ao vivo'}")
print(f"instrução com {len(PROMPT)} caracteres; ferramentas: {len(agente.tools)}")

modelo: gravação N1.1/falha
instrução com 2054 caracteres; ferramentas: 0


Para rodar, o `Runner` do ADK conduz o loop: entrega a mensagem ao agente, chama o modelo e devolve **eventos**. A sessão guarda a conversa.

In [4]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService  # ← N3.5: SqliteSessionService sobrevive à queda do processo
from google.genai import types

runner = Runner(app_name="rangoja", agent=agente, session_service=InMemorySessionService())
sessao = await runner.session_service.create_session(app_name="rangoja", user_id="101")
pergunta = types.Content(role="user", parts=[types.Part(text=mensagem)])

async for evento in runner.run_async(user_id="101", session_id=sessao.id, new_message=pergunta):
    for parte in (evento.content.parts if evento.content else []):
        if parte.function_call:
            print(f"[{evento.author}] chama {parte.function_call.name}({dict(parte.function_call.args)})")
        if parte.text:
            print(f"[{evento.author}] {parte.text}")

[unico] Lamento pela experiência que você teve. Seu pedido apresenta duas questões: 

1. Qualidade: O alimento chegou frio.
2. Item faltando: O refrigerante não foi entregue.

Para resolver, posso oferecer um voucher de R$ 15 pela qualidade do alimento e fazer o reembolso do valor do refrigerante. Você gostaria de seguir dessa forma?


**Parece certo, e é esse o problema.** A resposta segue a política (voucher de até R$ 15 por qualidade, reembolso do item que faltou). Mas olhe o que aconteceu de fato:

- **Nada foi consultado.** O agente acreditou no cliente: não tinha como saber se o refrigerante faltou mesmo, nem quanto custava.
- **Nada foi feito.** O cliente leu "reembolso", mas não existe reembolso no sistema: o agente tem zero ferramentas. Ao vivo, o modelo às vezes ainda diz que está "processando".
- **O código não tem o que executar.** A saída é texto: não há categoria nem valor que um sistema possa ler e cumprir.

Funciona na demo. Não funciona em produção.

Daqui em diante usamos um **atalho**: `atender(agente, caso)` faz o que as duas células acima fizeram (monta o agente e roda o `Runner`) e devolve o resultado em campos que o código pode verificar. Sem chave de API, `gravacao=` diz qual resposta gravada do modelo usar no lugar da OpenAI.

In [5]:
from rangoja.casos import caso
from rangoja.versoes import v0  # v0 = o agente que montamos acima: só o prompt, sem ferramentas

verificar(v0.construir().agente_raiz.instruction == PROMPT, "v0 é o agente montado acima (mesmo prompt)")

cliente = caso("C01")  # cliente 101, pedido 4821: "Meu pedido 4821 chegou frio e faltou o refri."
resultado = atender(v0, cliente, gravacao="N1.1/falha")  # a resposta que o gpt-4o-mini deu quando gravamos
print("decisão:             ", resultado.decisao)     # None: nada que o código possa ler
print("ferramentas chamadas:", resultado.trajetoria)  # []: o agente não consultou nem registrou nada
print("resposta veio de:    ", resultado.fonte)

decisão:              None
ferramentas chamadas: []
resposta veio de:     ['gravacao']


O modelo só sabe o que está no contexto. Texto plausível não é fato. Como dar fatos ao modelo? **Ferramentas.**

## N1.2 — Ferramentas trazem fatos

A mesma reclamação. Agora o agente ganha **ferramentas**: funções Python que o modelo pode pedir para executar, como `buscar_pedido` e `registrar_reembolso`. Primeiro, o que muda de um agente para o outro, lido dos próprios agentes:

In [6]:
sem_ferramentas = v0
com_ferramentas = v0.com("ferramentas")
comparar(sem_ferramentas, com_ferramentas)

Uma ferramenta é uma função Python comum. O modelo não vê o código: o ADK envia a ele só o **nome, a descrição (a docstring) e os parâmetros**, e o modelo decide se pede para chamá-la. É isto que ele recebe:

In [7]:
from google.adk.tools import FunctionTool

ferramentas_do_agente = {f.__name__: f for f in com_ferramentas.construir().agente_raiz.tools}
for nome, funcao in ferramentas_do_agente.items():
    print(f"{nome:22s} {funcao.__doc__}")

print("\no que o ADK envia ao modelo sobre buscar_pedido:")
print(FunctionTool(ferramentas_do_agente["buscar_pedido"])._get_declaration().model_dump_json(exclude_none=True, indent=1))

buscar_pedido          Consulta um pedido pelo número. Devolve itens, valores, pagamentos e atraso.
consultar_politica     Consulta um trecho da política de atendimento. Temas: categorias, item_faltando, item_errado, cobranca, atraso, qualidade, golpe, limites, recorrencia, privacidade, memoria, fora_do_escopo.
oferecer_cupom         Oferece um cupom de desconto ao cliente. Até R$ 15 por atendimento.
registrar_reembolso    Registra o reembolso de um pedido. Ação irreversível: informe o motivo (ex.: item_faltando, cobranca).
solicitar_reenvio      Solicita o reenvio de um item que veio errado.
escalar_para_humano    Encaminha o atendimento para um atendente humano, com um resumo do caso.
encaminhar_seguranca   Encaminha um relato de golpe ou fraude à equipe de segurança. Não gera compensação.

o que o ADK envia ao modelo sobre buscar_pedido:
{
 "description": "Consulta um pedido pelo número. Devolve itens, valores, pagamentos e atraso.",
 "name": "buscar_pedido",
 "parameters": {
  "pro

Agora o atendimento. As ferramentas agem num banco de pedidos e reembolsos; começamos com ele limpo.

In [8]:
Armazem.reset()  # banco de pedidos e reembolsos limpo para esta seção
resultado = atender(com_ferramentas, caso("C01"), gravacao="N1.2/correcao")  # a resposta que o gpt-4o-mini deu quando gravamos
resumo(resultado)

reembolsos = [c.args.get("valor") for c in resultado.trajetoria if c.ferramenta == "registrar_reembolso"]
print("consultou o pedido antes de agir?", chamou(resultado.trajetoria, "buscar_pedido"))
print("reembolsos registrados:", reembolsos, "| preço do refrigerante no pedido: R$ 8,00")

agente,ferramenta,argumentos,efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 5, 'motivo': 'item_faltando'}",✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '4821', 'valor': 10, 'motivo': 'qualidade'}",✅ com efeito


consultou o pedido antes de agir? False
reembolsos registrados: [5] | preço do refrigerante no pedido: R$ 8,00


**O prompt pede, o modelo não obedece, e o código não percebe.** O prompt ganhou a frase *"Consulte os fatos com as ferramentas antes de decidir. Nunca invente valores de pedido."* e o agente tem `buscar_pedido`. Mesmo assim, nenhuma das 6 execuções gravadas consultou o pedido, e nesta o modelo registrou R$ 5 de reembolso por um refrigerante de R$ 8. Ter ferramentas disponíveis **não garante** que o modelo as use. Isso muda nas próximas seções: com a decisão estruturada (N1.4) e o estado da conversa (N1.5), a versão de falha da N1.6 consultou o pedido em 5 de 6 execuções; com o contexto enxuto, em 6 de 6. Com 6 amostras, não dá para separar o efeito de cada componente.

## N1.3 — O loop por baixo

Um agente é um loop: o modelo pede uma ferramenta, a ferramenta responde, o modelo decide o próximo passo ou responde. Os eventos do ADK mostram isso.

In [9]:
from rangoja.observabilidade import anatomia
s = preparar_secao("N1.3")
a = atender(s.correcao, s.caso, gravacao="N1.2/correcao")
print(anatomia(a))
passos = [ev for ev in a.eventos if ev.content and ev.content.parts]
verificar(len(passos) >= 2, "há pelo menos dois passos: o modelo age e o sistema responde")

atendimento C01 (v0+ferramentas) — status: concluido
├─ agente unico
│   ├─ pede ferramenta registrar_reembolso({'pedido_id': '4821', 'valor': 5, 'motivo': 'item_faltando'})
│   ├─ pede ferramenta oferecer_cupom({'pedido_id': '4821', 'valor': 10, 'motivo': 'qualidade'})
│   ├─ resultado de registrar_reembolso: {'reembolso_id': 4, 'valor': 5, 'status': 'registrado'}
│   ├─ resultado de oferecer_cupom: {'cupom_id': 1, 'valor': 10, 'status': 'oferecido'}
│   └─ texto: 'Seu pedido já foi processado. Registramos um reembolso de R$ 5 pelo refrigerante que falto'
└─ fonte das respostas do modelo: gravacao


## N1.4 — Prosa × estrutura

Caso `C02` (atraso). O agente decide em **prosa**: o código não tem como extrair categoria e valor. A correção é um **contrato**: a decisão é estruturada (`Decisao`) e a resposta ao cliente continua em texto.

**O que muda da falha para a correção**, lido dos próprios agentes que o ADK vai rodar:

In [10]:
s = preparar_secao("N1.4")
comparar(s.falha, s.correcao)

In [11]:
falha = atender(s.falha, s.caso, gravacao=s.gravacoes["falha"])
resumo(falha); amostra(s.gravacoes["falha"])
verificar_falha(falha, falha.decisao is None, "em prosa, o código não recebe uma decisão utilizável")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '5310'},✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '5310', 'valor': 10, 'motivo': 'Atraso na entrega'}",✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '5310', 'valor': 7, 'motivo': 'atraso'}",✅ com efeito


In [12]:
from rangoja import tipos
mostrar(tipos.Decisao)
correcao = atender(s.correcao, s.caso, gravacao=s.gravacoes["correcao"])
resumo(correcao); amostra(s.gravacoes["correcao"])
verificar_falha(correcao, correcao.decisao is not None, "a decisão agora é um objeto tipado")
verificar_falha(correcao, correcao.decisao.solucao in __import__("rangoja.politica", fromlist=["x"]).solucoes_permitidas(correcao.decisao.categoria),
                "a solução decidida existe na política para a categoria")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '5310'},✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '5310', 'valor': 10, 'motivo': 'atraso na entrega'}",✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '5310', 'valor': 7, 'motivo': 'atraso'}",✅ com efeito


Nota: com a API do ADK, a saída estruturada chega por uma ferramenta interna (`set_model_response`). Ela aparece nos eventos e é ignorada nas funções de trajetória.

## N1.5 — Estado da conversa

Caso `C11`: três mensagens (sem pedido → o 4822, faltou o refri → "e a sobremesa também"). Sem estado, cada mensagem abre uma sessão nova.

**O que muda da falha para a correção**, lido dos próprios agentes que o ADK vai rodar:

In [13]:
s = preparar_secao("N1.5")
comparar(s.falha, s.correcao)

In [14]:
falha = atender(s.falha, s.caso, gravacao=s.gravacoes["falha"])
resumo(falha); amostra(s.gravacoes["falha"])
print("estado da última sessão (sem estado):", {k: v for k, v in falha.estado.items() if k in ("pedido_id", "compensacoes")})

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4822'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4822', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '4822', 'valor': 10, 'motivo': 'atraso'}",✅ com efeito
unico,escalar_para_humano,"{'motivo': 'Recorrência de reembolsos', 'resumo': 'Cliente solicitou reembolso por item faltante (refrigerante 2L) e já possui 3 reembolsos nos últimos 30 dias.'}",✅ com efeito


estado da última sessão (sem estado): {}


In [15]:
correcao = atender(s.correcao, s.caso, gravacao=s.gravacoes["correcao"])
resumo(correcao); amostra(s.gravacoes["correcao"])
print("estado da sessão (com estado):", {k: v for k, v in correcao.estado.items() if k in ("pedido_id", "compensacoes")})
verificar_falha(correcao, correcao.estado.get("pedido_id") == "4822", "o estado guardou o pedido da conversa")
verificar_falha(falha, falha.estado.get("pedido_id") != "4822", "sem estado, a última sessão não sabe qual é o pedido")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4822'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4822', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '4822', 'valor': 10, 'motivo': 'atraso na entrega'}",✅ com efeito


estado da sessão (com estado): {'pedido_id': '4822', 'compensacoes': 18.0}


## N1.6 — O que entra no contexto

Caso `C01`, agora quase na `v1`: ferramentas, decisão estruturada e estado, mas **sem o contexto enxuto**. `buscar_pedido` devolve o pedido inteiro, inclusive os dados do entregador, que o modelo não precisa para decidir.

**O que muda da falha para a correção**, lido dos próprios agentes que o ADK vai rodar:

In [16]:
s = preparar_secao("N1.6")
comparar(s.falha, s.correcao)

In [17]:
falha = atender(s.falha, s.caso, gravacao=s.gravacoes["falha"])
resumo(falha); amostra(s.gravacoes["falha"])
buscas = [c for c in falha.trajetoria if c.ferramenta == "buscar_pedido"]
if buscas:
    print("o que o modelo recebeu sobre o entregador:", buscas[0].resultado.get("entregador"))
    verificar_falha(falha, "entregador" in buscas[0].resultado, "nome, telefone e placa do entregador entraram no contexto do modelo")
else:
    print("nesta execução o modelo não consultou o pedido")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4821'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 0, 'motivo': 'item_faltando'}",⛔ sem efeito
unico,oferecer_cupom,"{'pedido_id': '4821', 'valor': 15, 'motivo': 'entrega de item com qualidade abaixo do esperado'}",✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito


o que o modelo recebeu sobre o entregador: {'entregador_id': '301', 'nome_completo': 'Ricardo Fictício Souza', 'telefone': '(00) 90000-1111', 'placa': 'ABC1D23'}


**A resposta está certa, e é por isso que o problema passa despercebido.** O dado do entregador não aparece na resposta, mas está no contexto: o modelo pode repeti-lo a qualquer momento. Em N2.3 simulamos esse vazamento (nas 10 execuções reais, o modelo não vazou) e vemos o guardrail de saída barrá-lo. Aqui a correção vem antes: a ferramenta devolve só o que a decisão precisa.

In [18]:
correcao = atender(s.correcao, s.caso, gravacao=s.gravacoes["correcao"])
resumo(correcao); amostra(s.gravacoes["correcao"])
buscas = [c for c in correcao.trajetoria if c.ferramenta == "buscar_pedido"]
if buscas:
    print("campos devolvidos ao modelo (versão enxuta):", sorted(buscas[0].resultado.keys()))
    verificar_falha(correcao, "entregador" not in buscas[0].resultado, "com o contexto enxuto, nenhum dado do entregador chega ao modelo")
else:
    print("nesta execução o modelo não consultou o pedido")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4821'},✅ com efeito
unico,consultar_politica,{'tema': 'item_faltando'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '4821', 'valor': 15, 'motivo': 'pedido chegou frio e item faltou'}",✅ com efeito


campos devolvidos ao modelo (versão enxuta): ['atraso_minutos', 'itens', 'pagamentos', 'pedido_id', 'restaurante', 'status', 'taxa_entrega', 'total']


## N1.7 — Lembrar do cliente

Caso `C13`: dois atendimentos do mesmo cliente. No primeiro ele diz que prefere a entrega na portaria. Sem memória, o segundo atendimento não sabe.

**O que muda da falha para a correção**, lido dos próprios agentes que o ADK vai rodar:

In [19]:
s = preparar_secao("N1.7")
comparar(s.falha, s.correcao)

In [20]:
falha = atender(s.falha, s.caso, gravacao=s.gravacoes["falha"])
resumo(falha); amostra(s.gravacoes["falha"])
verificar_falha(falha, "portaria" not in (falha.resposta or "").lower(), "sem memória, o segundo atendimento não menciona a portaria")

In [21]:
mostrar(ferramentas.anotar_observacao)
correcao = atender(s.correcao, s.caso, gravacao=s.gravacoes["correcao"])
resumo(correcao); amostra(s.gravacoes["correcao"])
print("observações guardadas para o cliente:", correcao.estado.get("user:observacoes"))
verificar_falha(correcao, "portaria" in (correcao.resposta or "").lower(), "com memória, o agente usa a preferência")

agente,ferramenta,argumentos,efeito
unico,consultar_politica,{'tema': 'entrega'},⛔ sem efeito
unico,anotar_observacao,{'texto': 'Cliente preferiu que a entrega fosse feita na portaria.'},✅ com efeito


observações guardadas para o cliente: ['Preferência de entrega na portaria do cliente.', 'Cliente preferiu que a entrega fosse feita na portaria.']


Atenção: **memória livre é uma porta de entrada**. No NB02 vemos o cliente usá-la para gravar um direito que não tem.

## N1.8 — Recapitulação

| Problema | Componente | Onde |
|---|---|---|
| O modelo inventa fatos | `ferramentas` | `rangoja/ferramentas.py` |
| Decisão em prosa | `schema` | `rangoja/tipos.py` (`Decisao`) |
| A conversa se perde | `estado` | `session.state` |
| Contexto cheio de dados que não servem | `contexto_enxuto` | `ferramentas._resumo_enxuto` |
| O agente não lembra do cliente | `memoria_livre` | `user:observacoes` |

**Ponte para o NB02:** o agente agora age e lembra. Nada ainda o impede de agir mal.